In [1]:
from genetic_programming import GeneticProgramming
from fitness_functions.Gymnax_fitness_function import GymFitnessFunction

import jax
import jax.numpy as jnp
import jax.random as jr

import brax
from brax import envs
import matplotlib.pyplot as plt

class BraxGymnaxWrapper:
    def __init__(self, env_name, backend):
        self.env = envs.get_environment(env_name=env_name, backend=backend)
        self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, key, state, action, params=None):

      action = action.reshape(self.action_space)
      next_state = self.env.step(state, action)
      done = next_state.done

      # Freeze state after done
      obs = jnp.where(done, state.obs, next_state.obs)
      new_state = jax.tree.map(
          lambda new, old: jnp.where(done, old, new),
          next_state,
          state
      )

      reward = jnp.where(done, 0.0, next_state.reward)

      return obs, new_state, reward, done, {}


These device(s) are detected: [CudaDevice(id=0)]
Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
key = jr.PRNGKey(0)

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = brax_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=1, max_nodes= 31)

/home/theun/yes/envs/modi_kozax/lib/python3.12/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


Input data should be formatted as: ['y0', 'y1', 'y2', 'y3'].


In [3]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.3 right
Compiling code for evaluation and evolution...
Finished compilation in 140.73 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -7.25, equation: [-0.0638]
Complexity: 2, fitness: -93.875, equation: [sin(y3)]
Complexity: 4, fitness: -233.375, equation: [y1 + sin(y1)]
Complexity: 5, fitness: -252.8125, equation: [3.42*y1]
Complexity: 7, fitness: -1000.0, equation: [2*y1 + y2 + y3]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -181.9375, equation: [y1]
Complexity: 3, fitness: -241.3125, equation: [2.42*y1]
Complexity: 5, fitness: -257.625, equation: [y1 + y2 + y3]
Complexity: 6, fitness: -554.375, equation: [y0 + y1 - sin(y0)]
Complexity: 7, fitness: -1000.0, equation: [2*y1 + y2 + y3]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -181.9375, equation: [y1]
Complexity: 3, fitness: -241.3125, equation: [2.42*y1]
Complexity: 5, fitness: -257.625, equation: [y1 + y2 + y3]
Complexity: 6, fitness: -554.37

In [4]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
key = jr.PRNGKey(0)

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = brax_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=1, max_nodes= 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7'].


In [5]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.3 right
Compiling code for evaluation and evolution...
Finished compilation in 117.34 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -184.10488891601562, equation: [-0.0638]
Complexity: 2, fitness: -440.75897216796875, equation: [sin(y1)]
Complexity: 3, fitness: -775.4652709960938, equation: [y2 - y7]
Complexity: 7, fitness: -1006.9361572265625, equation: [0.486*y7*(y7 - 1.94)]
Complexity: 9, fitness: -1246.233154296875, equation: [(y1 - 0.743)*(y2 + y4*y7)]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -424.4704895019531, equation: [y6]
Complexity: 2, fitness: -440.75897216796875, equation: [sin(y1)]
Complexity: 3, fitness: -815.1287231445312, equation: [y1 - y7]
Complexity: 7, fitness: -1006.9361572265625, equation: [0.486*y7*(y7 - 1.94)]
Complexity: 9, fitness: -1246.233154296875, equation: [(y1 - 0.743)*(y2 + y4*y7)]
Complexity: 15, fitness: -2016.470703125, equation: [(y0**2 + y2)*(y3*y6 + 2*y5 - 1.76)]
invalid s

In [2]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
key = jr.PRNGKey(0)

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = brax_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes= 31)

/home/theun/yes/envs/modi_kozax/lib/python3.12/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10'].


In [3]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.3 right
Compiling code for evaluation and evolution...
Finished compilation in 180.79 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: 637.1117553710938, equation: [0.639, 0]
Complexity: 2, fitness: 112.74197387695312, equation: [0, sin(y8)]
Complexity: 4, fitness: 112.74169921875, equation: [0, -sin(y6 - y8)]
Complexity: 5, fitness: 110.22703552246094, equation: [0, y1*y5 + y4 + y5]
Complexity: 9, fitness: 107.21183013916016, equation: [0, y5*(y0*y8 - y6 + y7)]
Complexity: 11, fitness: 95.11971282958984, equation: [y3*y5*(y10 + y8), (y0 + y3)*(y10 + y8)]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: 112.77143859863281, equation: [0, y8]
Complexity: 2, fitness: 112.74197387695312, equation: [0, sin(y8)]
Complexity: 3, fitness: 111.21009826660156, equation: [0, 0.247*y8]
Complexity: 5, fitness: 110.22703552246094, equation: [0, y1*y5 + y4 + y5]
Complexity: 9, fitness: 87.7440414428711, equation: [y3*y5*(y10 + y8), y0*(y1

In [ ]:
print(strategy.node_to_string)

In [2]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
key = jr.PRNGKey(0)

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = brax_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes= 31)

/home/theun/yes/envs/modi_kozax/lib/python3.12/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7'].


In [3]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.3 right
Compiling code for evaluation and evolution...
Finished compilation in 183.57 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: 7.452690124511719, equation: [0.839, 0]
Complexity: 2, fitness: -11.424619674682617, equation: [0, sin(y6)]
Complexity: 3, fitness: -23.778186798095703, equation: [0, y6 + y7]
Complexity: 4, fitness: -24.03884506225586, equation: [y2 + y4, sin(y4)]
Complexity: 5, fitness: -32.559181213378906, equation: [0, y0 + y1 + y5]
Complexity: 7, fitness: -131.67919921875, equation: [y0 + y6, y1*y6**2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -12.504685401916504, equation: [y7, 0]
Complexity: 3, fitness: -23.778186798095703, equation: [0, y6 + y7]
Complexity: 4, fitness: -37.31573486328125, equation: [2*y4, cos(y4)]
Complexity: 5, fitness: -108.44076538085938, equation: [y0 + y6, -y4 + y6]
Complexity: 7, fitness: -131.67919921875, equation: [y0 + y6, y1*y6**2]
Complexity: 13, fitness: -212.0331

In [4]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
key = jr.PRNGKey(0)

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = brax_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=3, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10'].


In [5]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Mujoco Modi (not updated) 0.3 right
Compiling code for evaluation and evolution...
Finished compilation in 119.86 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -212.62869262695312, equation: [0.639, 0, 0]
Complexity: 2, fitness: -220.23825073242188, equation: [0, cos(y0), 0]
Complexity: 5, fitness: -272.8374938964844, equation: [0, y2*(y1 - 0.875), 0]
Complexity: 10, fitness: -282.33673095703125, equation: [0, y3*(y2 + y3)*cos(y6), y1*y3]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -250.21563720703125, equation: [0, y4, 0]
Complexity: 3, fitness: -292.82208251953125, equation: [0, y3*y7, 0]
Complexity: 15, fitness: -325.93280029296875, equation: [y1*y9, 2*y1 + y2*y9 - y9 + cos(y10) - cos(y9), 0]
Complexity: 20, fitness: -767.7973022460938, equation: [y0*y3*(y1 + y3) + 2*y0*y3 + y0 + 2*sin(y6), y0*y1, y0*y10*y3 + y10*(-y0 + y6)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -250.319580078125, equation: [0, y4, 0]
Complexity: 3